# Labelling every pixel with OpenCV and the Selfie Segmenter

If you've used computer vision models before, you'll know that many draw a box or a set of points around the things they find. A segmentation model is different. It gives a label to *every single pixel* in the frame. That lets you recolor hair, swap in a background, or change the color of your clothes in real time.

In this notebook, we use MediaPipe's Selfie Multi-Class model in OpenCV, which sorts pixels into six classes:

| Class | What it covers |
|---|---|
| `background` | Everything behind the person |
| `hair` | Hair on the head |
| `body_skin` | Skin on arms and hands |
| `face_skin` | Skin on the face |
| `clothes` | Clothing |
| `others` | Glasses, hats, and other extras |

In this notebook, we will download the model from Hugging Face, load the model, create a segment function, and then start the webcam (running the segment function on each frame).

## Downloading the Selfie Segmenter model from Hugging Face

First, we need to download the model from Hugging Face. The following cell downloads a file called `model.onnx`. This file is a neural network model in ONNX format. ONNX stands for Open Neural Network eXchange and is one of the most popular formats for small models running in the browser or on hardware devices.

The file is about 16 MB, so the download may take a few moments.

In [ ]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id="senty-au/selfie_multiclass_256x256-ONNX", filename="onnx/model.onnx")

## Loading the model

With the ONNX file downloaded, we now load the model into OpenCV using the `readNet` method in OpenCV's `dnn` module.

In [ ]:
import cv2

selfie_net = cv2.dnn.readNet(model_path)
print("Neural network loaded")

## Defining the classes

The model doesn't return the name of what it sees at each pixel, only six scores. To fix this, we define an array of strings called `SEGMENT_CLASSES` which map these to their names. For example, if the highest score for a pixel is at position 1, we can use this array to know that the pixel is part of someone's hair.

In [ ]:
SEGMENT_CLASSES = ['background', 'hair', 'body_skin', 'face_skin', 'clothes', 'others']

## The segment function

The code in the next cell defines a new function called `segment`. After we connect to the webcam, we'll run this function continuously, passing in frames from the webcam for the model to label.

The function returns a dictionary of **masks**, one for each class. A mask is a small grayscale picture, the same shape as the frame, that is bright wherever the model thinks a pixel belongs to that class and dark everywhere else.

If you are interested in seeing how it works, expand the cell to view the code!

In [ ]:
import numpy as np

INPUT_SIZE = 256

def segment(frame):
  """Run the selfie segmenter on a BGR frame. Returns a dictionary with one
  256x256 mask per class in SEGMENT_CLASSES, e.g. masks['hair']. Each mask
  says how likely every pixel is to belong to that class, from 0 to 1."""
  # 1. Squash the whole frame to 256x256 (no letterbox: the model was
  #    trained on stretched images). It wants RGB pixels from 0 to 1, in
  #    height x width x channel order (not a blobFromImage blob).
  resized = cv2.resize(frame, (INPUT_SIZE, INPUT_SIZE))
  blob = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)[np.newaxis].astype(np.float32) / 255

  # 2. Run the network: six scores for each of the 256x256 pixels, one per
  #    class. A bigger score means "more likely to be this class".
  selfie_net.setInput(blob)
  scores = selfie_net.forward()[0]

  # 3. Softmax: turn each pixel's six scores into six probabilities that
  #    add up to 1.
  scores = np.exp(scores - scores.max(axis=2, keepdims=True))
  probabilities = scores / scores.sum(axis=2, keepdims=True)

  # 4. Split into one mask per class.
  return {name: probabilities[:, :, i] for i, name in enumerate(SEGMENT_CLASSES)}

## Start the webcam!

Now that everything is setup, we'll start the webcam, capture frames in a loop, and pass them to the segment function. We'll then pass the `hair` mask to the canvas, which tints those pixels pink on the screen. Make sure your face, hair, and shoulders are in view.

This model does a lot more work than a detector, so the color updates a few times a second and may trail slightly behind you when you move.

Once it is working, press **Stop** and try changing `'hair'` to another class such as `'clothes'` or `'background'`, or pick a different color. The `opacity` (from `0.0` to `1.0`) sets how strong the color is.

In [ ]:
from codetto import graphics2d, webcam

canvas = graphics2d.canvas()
camera = webcam.start(canvas, mirror=True)
try:
  while True:
    frame = camera.capture_array()
    masks = segment(frame)
    canvas.draw_mask(masks['hair'], color='#ff69b4', opacity=0.7)
finally:
  camera.stop()

## Layering two effects

You are not limited to one class per frame. `canvas.draw_masks` takes a list of layers, each with its own mask, color and opacity, and paints them all at once.

The cell below colors hair pink and clothes blue at the same time. Press **Stop** when you are done.

In [ ]:
from codetto import graphics2d, webcam

canvas = graphics2d.canvas()
camera = webcam.start(canvas, mirror=True)
try:
  while True:
    frame = camera.capture_array()
    masks = segment(frame)
    canvas.draw_masks([
      {'mask': masks['hair'], 'color': '#ff69b4', 'opacity': 0.7},
      {'mask': masks['clothes'], 'color': '#00bfff', 'opacity': 0.6},
    ])
finally:
  camera.stop()

## Replacing the background

A mask doesn't have to be filled with a color. Pass `image=` to `canvas.draw_mask` and the picture shows through wherever the mask is bright. Point it at the `background` mask and everything behind you is replaced, while you stay in the picture.

The image here comes from `samples.Images`, the set of pictures built into Codetto. Run the cell to drop yourself in front of a new background. Press **Stop** when you are done.

In [ ]:
from codetto import graphics2d, samples, webcam

canvas = graphics2d.canvas()
camera = webcam.start(canvas, mirror=True)
try:
  while True:
    frame = camera.capture_array()
    masks = segment(frame)
    canvas.draw_mask(masks['background'], image=samples.Images.Undersea, opacity=0.95)
finally:
  camera.stop()

# Check Your Understanding